# Logistic Regression

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## Where We Are

| Algorithm | Output type | Loss function | Activation |
|---|---|---|---|
| Perceptron | Hard 0 or 1 | None | Step function |
| Linear Regression | Any real number | MSE | None |
| **Logistic Regression** | **Probability (0–1)** | **Cross-entropy** | **Sigmoid** |

Logistic Regression combines the best of both:
- Like Linear Regression: uses a proper loss function and gradient descent
- Like the Perceptron: answers a classification question (win or not)
- Unlike either: gives you a **probability**, not just a label

---

## What is Logistic Regression?

The name is slightly misleading — despite saying "regression", it's a **classification algorithm**. The "regression" part refers to how it's trained (fitting a curve to data), not what it predicts.

The core question it answers:
> *"Given what we know about two teams, what is the **probability** that the home team wins?"*

That probability is the real output. The 0/1 label is just a decision made from it.

---

## The Full Pipeline

### Step 1: Weighted sum (same as always)

$$z = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b$$

z can be any real number — positive, negative, huge, tiny.

### Step 2: Sigmoid — converting z to a probability

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

The sigmoid function squashes z into the range (0, 1). It has a beautiful S-shape:

- z very negative → p ≈ 0 (almost certainly not a home win)
- z = 0 → p = 0.5 (total uncertainty — could go either way)
- z very positive → p ≈ 1 (almost certainly a home win)

The further z is from zero, the more confident the model is. The closer to zero, the more uncertain.

### Step 3: Decision threshold

To get a hard label, apply a threshold (default 0.5):
- p ≥ 0.5 → predict Home Win (1)
- p < 0.5 → predict Draw/Away (0)

But the probability is the real output. You could use any threshold — 0.6 if you only want high-confidence predictions, 0.4 if you want to catch more home wins even at the cost of more false positives.

### Step 4: Cross-entropy loss

$$L = -\frac{1}{n} \sum \left[ y \log(p) + (1-y) \log(1-p) \right]$$

This measures how surprised the model was by the correct answers.

| Situation | Loss |
|---|---|
| y=1, p=0.99 | ≈ 0.01 — confident and right, barely any loss |
| y=1, p=0.50 | ≈ 0.69 — uncertain, moderate loss |
| y=1, p=0.01 | ≈ 4.60 — confident and catastrophically wrong |

This is better than MSE for probabilities: it doesn't just measure distance, it specifically hammers confident wrong predictions.

### Step 5: Gradient descent (same as Linear Regression)

The gradient of cross-entropy turns out beautifully simple:

$$\frac{\partial L}{\partial w} = \frac{1}{n} X^T (p - y)$$

$(p - y)$ is the error — predicted probability minus true label. The gradient says: for each weight, how much did the model's overconfidence or underconfidence in that direction contribute to the error? Then we step downhill as usual.


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.logistic_regression import LogisticRegression

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Visualise the Sigmoid Function

Before touching the data, let's see what the sigmoid actually looks like.

In [ ]:
model_viz = LogisticRegression()
z_vals = np.linspace(-8, 8, 300)
p_vals = model_viz._sigmoid(z_vals)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z_vals, p_vals, color='#4878CF', linewidth=2.5)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='p = 0.5 (threshold)')
ax.axvline(0,   color='gray', linestyle='--', linewidth=1, label='z = 0')

# Annotate key points
for z, label in [(-6, 'p ≈ 0\n(very unlikely)'), (0, 'p = 0.5\n(uncertain)'), (6, 'p ≈ 1\n(very likely)')]:
    p = model_viz._sigmoid(np.array([z]))[0]
    ax.annotate(label, xy=(z, p), xytext=(z, p + 0.12),
                ha='center', fontsize=9,
                arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlabel('z  (weighted sum)', fontsize=12)
ax.set_ylabel('σ(z)  =  probability', fontsize=12)
ax.set_title('The Sigmoid Function', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.15)
plt.tight_layout()
plt.show()

This S-curve is the entire difference between Logistic Regression and Linear Regression. One extra function applied to z, and suddenly our output is a meaningful probability.

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Target: binary — home win (1) or draw/away win (0)
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate',
    'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches | Home win rate: {df_clean["home_win"].mean():.1%}')

---
## 3. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['home_win'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

---
## 4. Train the Model

In [ ]:
model = LogisticRegression(learning_rate=0.1, n_epochs=1000, random_state=SEED)
model.fit(X_train_sc, y_train)

print(f'Train accuracy:  {model.score(X_train_sc, y_train):.3f}')
print(f'Test  accuracy:  {model.score(X_test_sc,  y_test):.3f}')
print(f'Train log-loss:  {model.log_loss(X_train_sc, y_train):.4f}')
print(f'Test  log-loss:  {model.log_loss(X_test_sc,  y_test):.4f}')

---
## 5. Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model.loss_history_, color='#4878CF', linewidth=2)
ax.fill_between(range(len(model.loss_history_)), model.loss_history_, alpha=0.1, color='#4878CF')
ax.axhline(np.log(2), color='#E87272', linestyle='--', linewidth=1.2,
           label=f'Random baseline (log 2 ≈ {np.log(2):.3f})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('Logistic Regression — Loss Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

The red dashed line is the loss a model would get if it predicted 50% for every match — the "I have no idea" baseline. Our model should comfortably beat that.

Cross-entropy loss of log(2) ≈ 0.693 is the theoretical baseline for a random binary classifier. Any model worth training should finish well below it.

---
## 6. Predicted Probabilities

This is what makes Logistic Regression genuinely useful — not just 0/1, but a full distribution of confidence.

In [ ]:
probs = model.predict_proba(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution of predicted probabilities, split by true label
axes[0].hist(probs[y_test == 0], bins=40, alpha=0.6, color='#E87272',
             label='True: Draw/Away (0)', density=True)
axes[0].hist(probs[y_test == 1], bins=40, alpha=0.6, color='#4878CF',
             label='True: Home Win (1)', density=True)
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold 0.5')
axes[0].set_xlabel('Predicted Probability of Home Win', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Predicted Probability Distribution', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# Calibration: average predicted prob vs actual win rate in bins
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers, actual_rates = [], []
for i in range(n_bins):
    mask = (probs >= bin_edges[i]) & (probs < bin_edges[i+1])
    if mask.sum() > 0:
        bin_centers.append(probs[mask].mean())
        actual_rates.append(y_test[mask].mean())

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Perfect calibration')
axes[1].scatter(bin_centers, actual_rates, color='#4878CF', s=60, zorder=5)
axes[1].plot(bin_centers, actual_rates, color='#4878CF', linewidth=1.5, label='Model')
axes[1].set_xlabel('Mean Predicted Probability', fontsize=11)
axes[1].set_ylabel('Actual Win Rate', fontsize=11)
axes[1].set_title('Calibration Plot', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### Reading these plots

**Left — Probability distributions**: Ideally, the blue (true home wins) would be pushed right (high probability) and red (true draws/away) pushed left (low probability). Overlap in the middle is where the model is uncertain — and where football genuinely is unpredictable.

**Right — Calibration plot**: A perfectly calibrated model's line sits exactly on the diagonal. If the model says 70% probability, 70% of those matches should actually be home wins. Deviation from the diagonal tells us where the model is overconfident or underconfident.

---
## 7. Confusion Matrix & Classification Report

In [ ]:
y_pred = model.predict(X_test_sc)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'], ax=ax)
ax.set_title('Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Draw/Away', 'Home Win']))

---
## 8. ROC Curve

The ROC curve is a powerful tool unique to probability-based classifiers — you can't make one for the Perceptron because it has no probabilities.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#4878CF', linewidth=2, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.2, label='Random classifier (AUC = 0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Reading the ROC curve

The ROC curve answers: *"If I lower my threshold from 1.0 down to 0.0, how does my true positive rate (catching actual home wins) trade off against my false positive rate (wrongly predicting home wins)?"*

- **AUC = 1.0**: perfect classifier
- **AUC = 0.5**: random guessing (the diagonal)
- **AUC = 0.6–0.7**: modest predictive power — typical for football

The curve bowing toward the top-left corner is good. A curve sitting on the diagonal means the model learned nothing.

---
## 9. Feature Weights

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4878CF' if w > 0 else '#E87272' for w in model.weights_]
ax.barh(feature_cols, model.weights_, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Weight (log-odds scale)', fontsize=12)
ax.set_title('Learned Feature Weights', fontsize=13, fontweight='bold')

pos_patch = mpatches.Patch(color='#4878CF', label='Increases P(home win)')
neg_patch = mpatches.Patch(color='#E87272', label='Decreases P(home win)')
ax.legend(handles=[pos_patch, neg_patch], fontsize=9)
plt.tight_layout()
plt.show()

In Logistic Regression, weights operate on the **log-odds scale**. A positive weight means that feature makes a home win more likely; negative means less likely. The magnitude tells you how strongly.

---
## 10. Threshold Sensitivity

Since our output is a probability, we can choose any threshold — not just 0.5.

In [ ]:
thresholds = np.linspace(0.1, 0.9, 50)
accuracies, precisions, recalls = [], [], []

for t in thresholds:
    preds = (probs >= t).astype(int)
    accuracies.append(np.mean(preds == y_test))
    tp = np.sum((preds == 1) & (y_test == 1))
    fp = np.sum((preds == 1) & (y_test == 0))
    fn = np.sum((preds == 0) & (y_test == 1))
    precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
    recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, accuracies, label='Accuracy',  color='#4878CF', linewidth=2)
ax.plot(thresholds, precisions, label='Precision', color='#2ecc71', linewidth=2)
ax.plot(thresholds, recalls,    label='Recall',    color='#E87272', linewidth=2)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='Default threshold')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Metrics vs Decision Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

This plot reveals an important practical insight: **the best threshold depends on what you care about.**

- If you're building a betting model and wrong predictions are costly → raise the threshold, sacrifice recall for precision
- If you want to flag any possible home win → lower the threshold, catch more true positives at the cost of more false positives

The Perceptron can't do any of this — it has no probabilities, so the threshold is permanently fixed at z=0.

---
## 11. Discussion — Does Logistic Regression Suit Football?

### What it does well
- **Probability output**: far more honest than a hard label. Saying "65% home win" is more useful than just "home win".
- **Well-calibrated**: with enough data, the probabilities tend to reflect real frequencies.
- **Interpretable**: weights directly tell you which features push toward each outcome.
- **Proper loss function**: cross-entropy is the principled choice for probability estimation.

### What it still struggles with

1. **Still linear**: the decision boundary is still a hyperplane. Football outcomes depend on complex, non-linear interactions between features.
2. **Three outcomes ignored**: football has draws. We collapsed that into "not a home win", which loses real information.
3. **Feature engineering dependency**: the model is only as good as the features we hand it. It can't discover structure we didn't encode.
4. **Historical data only**: injuries, weather, motivation, referee — the model sees none of it.

### The progression so far

| | Perceptron | Linear Regression | Logistic Regression |
|---|---|---|---|
| Output | Hard 0/1 | Real number | Probability |
| Loss | None | MSE | Cross-entropy |
| Gradient descent | No | Yes | Yes |
| Confidence score | No | No | Yes |
| Suitable for football | Low | Moderate | Moderate-good |

Each step has been a genuine upgrade. Neural Networks will break the linearity constraint — that's where things get powerful.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Binary classification |
| **Output** | Probability of home win ∈ (0, 1) |
| **Activation** | Sigmoid σ(z) = 1/(1+e^{−z}) |
| **Loss function** | Binary cross-entropy (log loss) |
| **Optimization** | Batch Gradient Descent |
| **Key new capability** | Calibrated probability estimates |
| **Football suitability** | Moderate — linear boundary limits it |
| **Key concept introduced** | Probabilistic classification, ROC/AUC, calibration |
